##Importing Data

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU is enabled.")
    print("device count: {}, current device: {}".format(torch.cuda.device_count(), torch.cuda.current_device()))
else:
    print("GPU is not enabled.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU is enabled.
device count: 1, current device: 0


In [ ]:
%pip install datasets
%pip install seqeval
%pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
from google.colab import drive
import pickle
drive.mount('/content/drive')
with open('/content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/mBERT_tokenized_documents_trun.pkl', 'rb') as file:
  data = pickle.load(file)

Mounted at /content/drive


In [ ]:
from itertools import islice
test_data = {
    "train": list(islice(data["train"], 20)),
    "validation": list(islice(data["validation"], 20))
}

##Finetuning

In [ ]:
from transformers import BertForTokenClassification
from transformers import BertTokenizerFast
from torch.utils.data import Dataset, DataLoader
from datasets import Dataset
from itertools import product
from sklearn.metrics import classification_report, accuracy_score, f1_score
from seqeval.metrics import classification_report as seqeval_classification_report
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch.nn import CrossEntropyLoss
import numpy as np
import warnings
import os
import json

warnings.filterwarnings("ignore")

In [ ]:
class CustomDataset:
    def __init__(self, data):
        self.data = data
        self.train_data, self.val_data = self.data['train'], self.data['validation']

    def flatten(self, subset):
        flattened_data = []
        for item in subset:
            for j in range(len(item['input_ids'])):
                if len(item['input_ids'][j]) == 384:
                    flattened_data.append({
                        'input_ids': torch.tensor(item['input_ids'][j], dtype=torch.long),
                        'attention_mask': torch.tensor(item['attention_mask'][j], dtype=torch.long),
                        'labels': torch.tensor(item['labels'][j], dtype=torch.long),
                    })
        return flattened_data

    def create_data_loader(self, subset, batch_size = 5, shuffle=True, num_workers=0):
        flattened_subset = self.flatten(subset)
        loader_params = {
            'batch_size': batch_size,
            'shuffle': shuffle,
            'num_workers': num_workers
        }
        data_loader = DataLoader(flattened_subset, **loader_params)
        return data_loader

In [ ]:
class Trainer:
    def __init__(self, data, device='cuda'):
      custom_dataset = CustomDataset(data)
      self.training_loader = custom_dataset.create_data_loader(custom_dataset.train_data)
      self.validation_loader = custom_dataset.create_data_loader(custom_dataset.val_data)
      self.device = device
      self.id2label = {0: 'O', 1: 'B-ENTITY',2: 'I-ENTITY'}
      self.label2id = {'O': 0, 'B-ENTITY': 1, 'I-ENTITY': 2}
      self.best_params = None

    def train_epoch(self, idx1 = 384):
        tr_loss, tr_f1_score = 0, 0
        nb_tr_steps = 0
        tr_preds, tr_labels = [], []
        self.model.train()

        for idx, batch in enumerate(self.training_loader):
            ids = batch['input_ids'].to(self.device, dtype=torch.long)
            mask = batch['attention_mask'].to(self.device, dtype=torch.long)
            targets = batch['labels'].to(self.device, dtype=torch.long)

            outputs = self.model(input_ids=ids, attention_mask=mask, labels=targets)
            tr_logits = outputs.logits
            first_logits = tr_logits[:, :idx1, :]
            first_targets = targets[:, :idx1]
            flattened_logits = first_logits.reshape(-1, self.model.num_labels)
            flattened_targets = first_targets.reshape(-1)
            flattened_predictions = torch.argmax(flattened_logits, axis=1)

            nb_tr_steps += 1

            active = mask[:, :idx1].reshape(-1) == 1
            logits = flattened_logits[active]
            targets = torch.masked_select(flattened_targets, active)
            predictions = torch.masked_select(flattened_predictions, active)

            loss = self.criterion(logits, targets)
            tr_loss += loss.item()

            tr_preds.extend(predictions.cpu())
            tr_labels.extend(targets.cpu())

            tmp_tr_f1_score = f1_score(targets.cpu().numpy(), predictions.cpu().numpy(), average = 'macro')
            tr_f1_score += tmp_tr_f1_score

            torch.nn.utils.clip_grad_norm_(
                parameters=self.model.parameters(), max_norm = self.max_grad_norm
            )

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            #self.scheduler.step()

        epoch_loss = tr_loss / nb_tr_steps
        tr_f1_score = tr_f1_score / nb_tr_steps
        f1 = f1_score(tr_labels, tr_preds, average='macro')
        wandb.log({"Epoch": self.epoch + 1, "Train Loss": epoch_loss, "Batch Train F1 ": tr_f1_score, 'Overall Train F1': f1})

        return epoch_loss, tr_f1_score

    def evaluate(self, loader, idx2):
        self.model.eval()
        eval_loss, eval_f1_score = 0, 0
        nb_eval_steps = 0
        eval_preds, eval_labels = [], []

        with torch.no_grad():
            for idx, batch in enumerate(loader):

                ids = batch['input_ids'].to(self.device, dtype = torch.long)
                mask = batch['attention_mask'].to(self.device, dtype = torch.long)
                targets = batch['labels'].to(self.device, dtype = torch.long)

                outputs = self.model(input_ids=ids, attention_mask=mask, labels=targets)
                eval_logits = outputs.logits

                first_logits = eval_logits[:, :idx2, :]
                first_targets = targets[:, :idx2]

                flattened_logits = first_logits.reshape(-1, self.model.num_labels)
                flattened_targets = first_targets.reshape(-1)
                flattened_predictions = torch.argmax(flattened_logits, axis=1)

                nb_eval_steps += 1

                active = mask[:, :idx2].reshape(-1) == 1
                logits = flattened_logits[active]
                targets = torch.masked_select(flattened_targets, active)
                predictions = torch.masked_select(flattened_predictions, active)

                loss = self.criterion(logits, targets)
                eval_loss += loss.item()

                eval_labels.extend(targets)
                eval_preds.extend(predictions)

                tmp_eval_f1_score = f1_score(targets.cpu().numpy(), predictions.cpu().numpy(), average = 'macro')
                eval_f1_score += tmp_eval_f1_score

        labels = [self.id2label[id.item()] for id in eval_labels]
        predictions = [self.id2label[id.item()] for id in eval_preds]
        f1 = f1_score(labels, predictions, average = 'macro')

        eval_loss = eval_loss / nb_eval_steps
        eval_f1_score = eval_f1_score / nb_eval_steps
        wandb.log({"Epoch": self.epoch + 1, "Eval Loss": eval_loss, 'Batch Eval F1': eval_f1_score,  'Overall Eval F1': f1})

        return eval_loss, eval_f1_score, labels, predictions, f1

    def train(self, hyperparameter_grid):
        self.best_model_state_dict = None
        self.best_cross_entropy = float('inf')
        self.best_f1 = 0
        self.best_params = None
        self.model = BertForTokenClassification.from_pretrained('bert-base-multilingual-cased',
                                                  num_labels=len(self.id2label),
                                                  id2label=self.id2label,
                                                  label2id=self.label2id)
        initial_model_state_dict = self.model.state_dict()
        torch.save(initial_model_state_dict, "initial_model_weights.pth")

        for i, params in enumerate(product(*hyperparameter_grid.values())):
            hyperparameters = dict(zip(hyperparameter_grid.keys(), params))
            print('-------------------------------------------------------------------------')
            print('Hyperparameters: ', hyperparameters)
            print('-------------------------------------------------------------------------')

            wandb.init(
                          project="ELCardioCC_NER_mBERT",
                          name= f"lr{hyperparameters['learning_rate']}",
                          config=hyperparameters
                      )

            self.model.load_state_dict(initial_model_state_dict)
            self.model.to(self.device)
            self.learning_rate = hyperparameters['learning_rate']
            self.weight_decay = hyperparameters['weight_decay']
            self.optimizer = hyperparameters['optimizer'](self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
            #self.scheduler = hyperparameters['scheduler'](self.optimizer, step_size=2, gamma=0.1)
            self.criterion = hyperparameters['criterion']()
            self.max_grad_norm =  hyperparameters['max_grad_norm']
            self.num_epochs = hyperparameters['num_epochs']
            previous_eval_loss = float('inf')
            patience = 0

            trial_best_cross_entropy = float('inf')
            trial_best_f1 = 0
            trial_best_params = None

            for epoch in range(self.num_epochs):
                self.epoch = epoch
                epoch_loss, tr_f1_score = self.train_epoch(384)
                eval_loss, eval_f1_score, labels, predictions, f1 = self.evaluate(self.validation_loader, 384)
                print(f"Epoch {epoch + 1}/{self.num_epochs} => Train Loss: {epoch_loss:.4f}, Validation Loss: {eval_loss:.4f}")

                if eval_loss < trial_best_cross_entropy:
                    trial_best_model_state_dict = self.model.state_dict()
                    trial_best_params = hyperparameters
                    trial_best_params['best_epoch'] = epoch + 1
                    trial_best_cross_entropy = eval_loss
                    trial_best_f1 = f1

                if eval_loss > previous_eval_loss:
                    patience += 1
                    if patience >= 3:
                        print(f"Early stopping at epoch {epoch + 1}")
                        wandb.log({"early_stopping_epoch": epoch + 1})
                        break
                else:
                   patience = 0
                previous_eval_loss = eval_loss
            trial_best_model_info = {
                          'model_state_dict': trial_best_model_state_dict,
                          'hyperparameters': trial_best_params
                      }

            if trial_best_cross_entropy < self.best_cross_entropy:
                self.best_cross_entropy = trial_best_cross_entropy
                self.best_f1 = trial_best_f1
                self.best_params = trial_best_params
                self.best_model_state_dict = trial_best_model_state_dict
            wandb.finish()

        best_model_info = {
                              'model_state_dict': self.best_model_state_dict,
                              'hyperparameters': self.best_params
                          }

        print('-------------------------------------------------------------------------')
        print('Model Results:')
        print('-------------------------------------------------------------------------')
        print(f"Best Hyperparameters: {self.best_params}")
        print(f"Best Validation Loss: {self.best_cross_entropy}")

    def save_best_model(self, save_path, hyperparameter_grid):
        self.train(hyperparameter_grid)

        os.makedirs(save_path, exist_ok=True)
        self.model.load_state_dict(self.best_model_state_dict)
        self.model.to(self.device)

        self.model.save_pretrained(save_path)
        tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")
        tokenizer.save_pretrained(save_path)

        print(f"Best model saved to {save_path}")

        return self.model

##Run

In [ ]:
8e-6, 9e-6, 1e-5, 2e-5, 3e-5, 4e-5

In [ ]:
hyperparameters = {
    'learning_rate': [8e-6],
    'criterion': [CrossEntropyLoss],
    'optimizer': [AdamW],
    'max_grad_norm': [3],
    'num_epochs': [5],
    'batch_size': [4],
    'weight_decay' : [0.01]
}

In [ ]:
import torch
import wandb
wandb.login(key='b2d79908e1f8955784d351ca9b1bbcae22ea1769')
torch.manual_seed(42)
np.random.seed(42)
trainer = Trainer(data)
trainer.train(hyperparameters)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


-------------------------------------------------------------------------
Hyperparameters:  {'learning_rate': 8e-06, 'criterion': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'max_grad_norm': 3, 'num_epochs': 5, 'batch_size': 4, 'weight_decay': 0.01}
-------------------------------------------------------------------------


Epoch 1/5 => Train Loss: 0.1138, Validation Loss: 0.0731
Epoch 2/5 => Train Loss: 0.0620, Validation Loss: 0.0728
Epoch 3/5 => Train Loss: 0.0476, Validation Loss: 0.0666
Epoch 4/5 => Train Loss: 0.0372, Validation Loss: 0.0735
Epoch 5/5 => Train Loss: 0.0303, Validation Loss: 0.0761


Batch Eval F1,▁▆▆█▅
Batch Train F1,▁▅▇▇█
Epoch,▁▁▃▃▅▅▆▆██
Eval Loss,▆▆▁▆█
Overall Eval F1,▁▆▃█▅
Overall Train F1,▁▅▆▇█
Train Loss,█▄▂▂▁
Batch Eval F1,0.86339
Batch Train F1,0.92414
Epoch,5
Eval Loss,0.07607


-------------------------------------------------------------------------
Model Results:
-------------------------------------------------------------------------
Best Hyperparameters: {'learning_rate': 8e-06, 'criterion': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'max_grad_norm': 3, 'num_epochs': 5, 'batch_size': 4, 'weight_decay': 0.01, 'best_epoch': 3}
Best Validation Loss: 0.06663368256321414


##Best Model Saving

In [ ]:
best_hyperparameters = {
    'learning_rate': [8e-6],
    'criterion': [CrossEntropyLoss],
    'optimizer': [AdamW],
    'max_grad_norm': [3],
    'num_epochs': [5],
    'batch_size': [4],
    'weight_decay' : [0.01]}

In [ ]:
import torch
import wandb
wandb.login(key='b2d79908e1f8955784d351ca9b1bbcae22ea1769')
torch.manual_seed(42)
np.random.seed(42)
trainer = Trainer(data)
trainer.save_best_model("/content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/best_configuration_mbert", best_hyperparameters)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


-------------------------------------------------------------------------
Hyperparameters:  {'learning_rate': 8e-06, 'criterion': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'max_grad_norm': 3, 'num_epochs': 5, 'batch_size': 4, 'weight_decay': 0.01}
-------------------------------------------------------------------------


Epoch 1/5 => Train Loss: 0.1138, Validation Loss: 0.0731
Epoch 2/5 => Train Loss: 0.0620, Validation Loss: 0.0728
Epoch 3/5 => Train Loss: 0.0476, Validation Loss: 0.0666
Epoch 4/5 => Train Loss: 0.0372, Validation Loss: 0.0735
Epoch 5/5 => Train Loss: 0.0303, Validation Loss: 0.0761


Batch Eval F1,▁▆▆█▅
Batch Train F1,▁▅▇▇█
Epoch,▁▁▃▃▅▅▆▆██
Eval Loss,▆▆▁▆█
Overall Eval F1,▁▆▃█▅
Overall Train F1,▁▅▆▇█
Train Loss,█▄▂▂▁
Batch Eval F1,0.86339
Batch Train F1,0.92414
Epoch,5
Eval Loss,0.07607


-------------------------------------------------------------------------
Model Results:
-------------------------------------------------------------------------
Best Hyperparameters: {'learning_rate': 8e-06, 'criterion': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'max_grad_norm': 3, 'num_epochs': 5, 'batch_size': 4, 'weight_decay': 0.01, 'best_epoch': 3}
Best Validation Loss: 0.06663368256321414
Best model saved to /content/drive/MyDrive/Baseline_ELCardioCC_data/model_training/best_configuration_mbert


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-1